# Evaluate Artifact Emotion Models

This notebook evaluates and visualizes the saved models that already exist in `artifacts/models`.

Included models:
- MobileNetV2
- EfficientNetB0
- ResNet50

Excluded model:
- ResEmoteNet

The notebook rebuilds or loads the same deterministic test split, evaluates each available `.keras` model, displays comparison metrics, plots confusion matrices, shows per-class metric heatmaps, and displays sample predictions. It does not export CSV, TXT, or PNG files.


In [ ]:
import sys
import subprocess

# In Colab/Kaggle this installs only missing runtime helpers. Locally, comment this cell if the environment is already prepared.
packages = [
    'kagglehub',
    'opencv-python-headless',
    'scikit-learn',
    'seaborn',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])


In [ ]:
from pathlib import Path
import os
import json
import time
import shutil

import cv2
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

try:
    from IPython.display import display
except ImportError:
    display = print

sns.set_theme(style='whitegrid')
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


In [ ]:
SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
VALID_SIZE = 0.15
TEST_SIZE = 0.15

PROJECT_ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODEL_DIR = PROJECT_ROOT / 'artifacts' / 'models'
YOLO_CROP_DIR = PROCESSED_DIR / 'yolo_face_crops'

for path in [RAW_DIR, PROCESSED_DIR, MODEL_DIR, YOLO_CROP_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# ResEmoteNet is intentionally excluded from this evaluation notebook.
EXCLUDED_MODELS = ['ResEmoteNet']

MODEL_SPECS = {
    'MobileNetV2': {
        'path': MODEL_DIR / 'mobilenetv2' / 'best_mobilenetv2_emotion_model.keras',
        'metadata': MODEL_DIR / 'mobilenetv2' / 'metadata_mobilenetv2_mediapipe_v2.json',
        'color': '#4C78A8',
    },
    'EfficientNetB0': {
        'path': MODEL_DIR / 'EfficientNetB0' / 'best_efficientnetb0_emotion_model.keras',
        'metadata': MODEL_DIR / 'EfficientNetB0' / 'metadata_efficientnetb0_mediapipe.json',
        'color': '#54A24B',
    },
    'ResNet50': {
        'path': MODEL_DIR / 'Resnet50' / 'best_resnet50_emotion_model.keras',
        'metadata': MODEL_DIR / 'Resnet50' / 'metadata_resnet50_mediapipe_v3.json',
        'color': '#F58518',
    },
}

KAGGLE_DATASETS = ['mstjebashazida/affectnet', 'fatihkgg/affectnet-yolo-format']
LOCAL_DATASET_ROOTS = []
COPY_DATASETS_TO_RAW = False
USE_EXISTING_TEST_SPLIT = True

CLASS_NAMES = ['anger', 'contempt', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

AFFECTNET_ID_TO_LABEL = {
    0: 'anger',
    1: 'contempt',
    2: 'disgust',
    3: 'fear',
    4: 'happy',
    5: 'neutral',
    6: 'sad',
    7: 'surprise',
}

LABEL_ALIASES = {
    'anger': 'anger', 'angry': 'anger', 'angriness': 'anger', 'angry_face': 'anger', 'angryface': 'anger', 'mad': 'anger',
    'contempt': 'contempt', 'contemptuous': 'contempt',
    'disgust': 'disgust', 'disgusted': 'disgust', 'disgusting': 'disgust',
    'fear': 'fear', 'fearful': 'fear', 'scared': 'fear', 'afraid': 'fear',
    'happy': 'happy', 'happiness': 'happy', 'joy': 'happy', 'joyful': 'happy', 'smile': 'happy', 'smiling': 'happy',
    'neutral': 'neutral', 'normal': 'neutral', 'calm': 'neutral',
    'sad': 'sad', 'sadness': 'sad', 'unhappy': 'sad',
    'surprise': 'surprise', 'surprised': 'surprise', 'surprize': 'surprise',
}

AFFECTNET_LABEL_COLUMNS = ['expression', 'exp', 'emotion', 'label', 'class', 'category', 'emotion_label', 'expression_label']
AFFECTNET_PATH_COLUMNS = ['subDirectory_filePath', 'subdirectory_filepath', 'filepath', 'file_path', 'path', 'image', 'image_path', 'filename', 'file']

print('Project root:', PROJECT_ROOT)
print('Model dir:', MODEL_DIR)
print('Evaluated models:', list(MODEL_SPECS))
print('Excluded models:', EXCLUDED_MODELS)


In [ ]:
metadata_rows = []
for model_name, spec in MODEL_SPECS.items():
    metadata_path = Path(spec['metadata'])
    row = {
        'model': model_name,
        'model_exists': Path(spec['path']).exists(),
        'model_path': str(spec['path']),
        'metadata_exists': metadata_path.exists(),
        'metadata_path': str(metadata_path),
    }
    if metadata_path.exists():
        with open(metadata_path, 'r', encoding='utf-8') as f:
            metadata = json.load(f)
        row.update({
            'metadata_test_accuracy': metadata.get('test_accuracy'),
            'train_samples': metadata.get('train_samples'),
            'valid_samples': metadata.get('valid_samples'),
            'test_samples': metadata.get('test_samples'),
            'face_detector': metadata.get('face_detector'),
        })
    metadata_rows.append(row)

metadata_df = pd.DataFrame(metadata_rows)
display(metadata_df)


In [ ]:
def is_colab_runtime():
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


def is_kaggle_runtime():
    return Path('/kaggle/input').exists()


def find_kaggle_input_roots():
    input_dir = Path('/kaggle/input')
    if not input_dir.exists():
        return []
    return [p.resolve() for p in input_dir.iterdir() if p.is_dir()]


def find_local_raw_roots(raw_dir):
    if not raw_dir.exists():
        return []
    return [p.resolve() for p in raw_dir.iterdir() if p.is_dir()]


def prepare_kaggle_credentials():
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_json = kaggle_dir / 'kaggle.json'
    if kaggle_json.exists() or os.environ.get('KAGGLE_USERNAME'):
        return
    if is_kaggle_runtime():
        print('Kaggle runtime detected. Prefer adding the datasets as Inputs.')
        return
    if is_colab_runtime():
        print('Upload kaggle.json from your Kaggle account settings.')
        from google.colab import files  # type: ignore
        uploaded = files.upload()
        if 'kaggle.json' not in uploaded:
            raise FileNotFoundError('kaggle.json was not uploaded.')
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        kaggle_json.write_bytes(uploaded['kaggle.json'])
        try:
            os.chmod(kaggle_json, 0o600)
        except OSError:
            pass
        return
    raise FileNotFoundError('No Kaggle credentials found. Put kaggle.json in ~/.kaggle or set KAGGLE_USERNAME/KAGGLE_KEY.')


def download_kaggle_datasets(dataset_slugs, raw_dir, copy_to_raw=False):
    if LOCAL_DATASET_ROOTS:
        roots = [Path(p).expanduser().resolve() for p in LOCAL_DATASET_ROOTS]
        missing = [str(p) for p in roots if not p.exists()]
        if missing:
            raise FileNotFoundError('LOCAL_DATASET_ROOTS not found: ' + ', '.join(missing))
        print('Using LOCAL_DATASET_ROOTS:', roots)
        return roots

    local_roots = find_local_raw_roots(raw_dir)
    if local_roots:
        print('Using dataset folders already present in data/raw:', local_roots)
        return local_roots

    kaggle_input_roots = find_kaggle_input_roots()
    if kaggle_input_roots:
        print('Using Kaggle Input roots:', kaggle_input_roots)
        return kaggle_input_roots

    prepare_kaggle_credentials()
    downloaded_paths = []
    for slug in dataset_slugs:
        print(f'Downloading or locating Kaggle dataset: {slug}')
        dataset_path = Path(kagglehub.dataset_download(slug)).resolve()
        if copy_to_raw:
            target_path = raw_dir / slug.replace('/', '__')
            if target_path.exists():
                shutil.rmtree(target_path)
            shutil.copytree(dataset_path, target_path)
            dataset_path = target_path.resolve()
        downloaded_paths.append(dataset_path)
    return downloaded_paths


dataset_roots = download_kaggle_datasets(KAGGLE_DATASETS, RAW_DIR, COPY_DATASETS_TO_RAW)
print('Dataset roots:')
for root in dataset_roots:
    print('-', root)


In [ ]:
def normalize_label(value, numeric_mapping='affectnet'):
    if pd.isna(value):
        return None
    text = str(value).strip().lower().replace(' ', '_')
    if text in {'', 'nan', 'none', 'uncertain', 'non_face', 'non-face'}:
        return None
    try:
        numeric = int(float(text))
        if numeric_mapping == 'affectnet':
            return AFFECTNET_ID_TO_LABEL.get(numeric)
    except ValueError:
        pass
    return LABEL_ALIASES.get(text)


def infer_label_from_path(path):
    parts = [p.lower().replace(' ', '_') for p in Path(path).parts]
    for part in reversed(parts):
        label = normalize_label(part, numeric_mapping='affectnet')
        if label is not None:
            return label
    return None


def first_matching_column(columns, candidates):
    normalized = {str(col).strip().lower(): col for col in columns}
    for candidate in candidates:
        if candidate.lower() in normalized:
            return normalized[candidate.lower()]
    return None


def find_existing_image(root, value):
    candidate = Path(str(value).strip())
    candidates = []
    if candidate.is_absolute():
        candidates.append(candidate)
    else:
        clean_value = str(value).strip().lstrip('/\')
        candidates.extend([
            Path(root) / candidate,
            Path(root) / clean_value,
            Path(root) / 'Manually_Annotated' / 'Manually_Annotated_Images' / clean_value,
            Path(root) / 'Automatically_Annotated' / 'Automatically_Annotated_Images' / clean_value,
        ])
    for item in candidates:
        if item.exists() and item.suffix.lower() in IMAGE_EXTS:
            return item.resolve()
    return None


def collect_affectnet_csv_records(root):
    records = []
    for csv_path in Path(root).rglob('*.csv'):
        try:
            sample = pd.read_csv(csv_path, nrows=5)
        except Exception:
            continue
        path_col = first_matching_column(sample.columns, AFFECTNET_PATH_COLUMNS)
        label_col = first_matching_column(sample.columns, AFFECTNET_LABEL_COLUMNS)
        if path_col is None or label_col is None:
            continue
        print('Reading annotations:', csv_path)
        for chunk in pd.read_csv(csv_path, usecols=[path_col, label_col], chunksize=50000):
            chunk = chunk.dropna(subset=[path_col, label_col])
            for row in chunk.itertuples(index=False):
                image_value, label_value = row
                label = normalize_label(label_value, numeric_mapping='affectnet')
                if label is None:
                    continue
                image_path = find_existing_image(root, image_value)
                if image_path is None:
                    continue
                records.append({'path': str(image_path), 'label': label, 'source': Path(root).name, 'annotation_file': csv_path.name})
    return records


def collect_folder_label_records(root):
    records = []
    for file_path in Path(root).rglob('*'):
        if file_path.suffix.lower() not in IMAGE_EXTS:
            continue
        label = infer_label_from_path(file_path.relative_to(root))
        if label is not None:
            records.append({'path': str(file_path.resolve()), 'label': label, 'source': Path(root).name, 'annotation_file': None})
    return records


In [ ]:
def parse_yolo_names(root):
    import ast
    import re
    yaml_files = list(Path(root).rglob('data.yaml')) + list(Path(root).rglob('*.yaml'))
    for yaml_path in yaml_files:
        text = yaml_path.read_text(encoding='utf-8', errors='ignore')
        inline = re.search(r'^\s*names\s*:\s*(\[.*\]|\{.*\})\s*$', text, flags=re.MULTILINE)
        if inline:
            try:
                value = ast.literal_eval(inline.group(1))
                names = [value[i] for i in sorted(value)] if isinstance(value, dict) else list(value)
                return [normalize_label(name) or str(name).strip().lower() for name in names]
            except Exception:
                pass
        lines = text.splitlines()
        for idx, line in enumerate(lines):
            if line.strip().startswith('names:'):
                names = []
                for child in lines[idx + 1:]:
                    if not child.startswith((' ', '\t', '-')):
                        break
                    cleaned = child.strip().strip(',')
                    if not cleaned:
                        continue
                    if ':' in cleaned:
                        cleaned = cleaned.split(':', 1)[1].strip()
                    if cleaned.startswith('-'):
                        cleaned = cleaned[1:].strip()
                    cleaned = cleaned.strip('"'')
                    label = normalize_label(cleaned) or cleaned.lower()
                    names.append(label)
                if names:
                    return names
    return CLASS_NAMES


def find_image_for_label(label_path, images_dir):
    stem = label_path.stem
    for ext in IMAGE_EXTS:
        candidate = images_dir / f'{stem}{ext}'
        if candidate.exists():
            return candidate
    return None


def yolo_to_pixel_box(values, width, height):
    x_center, y_center, box_width, box_height = values
    x1 = int((x_center - box_width / 2) * width)
    y1 = int((y_center - box_height / 2) * height)
    x2 = int((x_center + box_width / 2) * width)
    y2 = int((y_center + box_height / 2) * height)
    x1 = max(0, min(width - 1, x1))
    y1 = max(0, min(height - 1, y1))
    x2 = max(x1 + 1, min(width, x2))
    y2 = max(y1 + 1, min(height, y2))
    return x1, y1, x2, y2


def collect_yolo_records(root):
    root = Path(root)
    yolo_names = parse_yolo_names(root)
    records = []
    label_dirs = [p for p in root.rglob('labels') if p.is_dir()]
    for labels_dir in label_dirs:
        images_dir = labels_dir.parent / 'images'
        if not images_dir.exists():
            continue
        for label_path in labels_dir.glob('*.txt'):
            image_path = find_image_for_label(label_path, images_dir)
            if image_path is None:
                continue
            try:
                image = Image.open(image_path).convert('RGB')
            except Exception:
                continue
            width, height = image.size
            lines = label_path.read_text(encoding='utf-8', errors='ignore').splitlines()
            for box_idx, line in enumerate(lines):
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                try:
                    class_id = int(float(parts[0]))
                    values = [float(v) for v in parts[1:5]]
                except ValueError:
                    continue
                if class_id < 0 or class_id >= len(yolo_names):
                    continue
                label = yolo_names[class_id]
                if label not in CLASS_TO_ID:
                    continue
                x1, y1, x2, y2 = yolo_to_pixel_box(values, width, height)
                crop = image.crop((x1, y1, x2, y2))
                split_name = labels_dir.parent.name
                target_dir = YOLO_CROP_DIR / Path(root).name / split_name / label
                target_dir.mkdir(parents=True, exist_ok=True)
                target_path = target_dir / f'{image_path.stem}_{box_idx}{image_path.suffix.lower()}'
                if not target_path.exists():
                    crop.resize(IMG_SIZE).save(target_path, quality=95)
                records.append({'path': str(target_path.resolve()), 'label': label, 'source': Path(root).name, 'annotation_file': str(label_path.relative_to(root))})
    return records


def collect_image_files(roots):
    records = []
    for root in roots:
        root = Path(root)
        yolo_records = collect_yolo_records(root)
        csv_records = collect_affectnet_csv_records(root)
        folder_records = collect_folder_label_records(root)
        records.extend(yolo_records)
        records.extend(csv_records)
        records.extend(folder_records)
    df = pd.DataFrame(records)
    if df.empty:
        return df
    df = df.drop_duplicates(subset=['path']).reset_index(drop=True)
    df = df[df['label'].isin(CLASS_NAMES)].copy()
    df['label_id'] = df['label'].map(CLASS_TO_ID).astype(int)
    return df


In [ ]:
test_split_path = PROCESSED_DIR / 'test_split_v2.csv'

if USE_EXISTING_TEST_SPLIT and test_split_path.exists():
    print('Using existing test split:', test_split_path)
    test_df = pd.read_csv(test_split_path)
else:
    df_images = collect_image_files(dataset_roots)
    print('Images found:', len(df_images))
    if df_images.empty:
        raise ValueError('No valid emotion images found.')
    display(df_images['label'].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int).to_frame('count'))
    min_class_count = int(df_images['label'].value_counts().min())
    if min_class_count < 2:
        raise ValueError('Each class needs at least 2 samples for stratified splitting.')
    train_df, temp_df = train_test_split(df_images, test_size=VALID_SIZE + TEST_SIZE, stratify=df_images['label'], random_state=SEED)
    relative_test = TEST_SIZE / (VALID_SIZE + TEST_SIZE)
    valid_df, test_df = train_test_split(temp_df, test_size=relative_test, stratify=temp_df['label'], random_state=SEED)
    print('Created deterministic train/valid/test split in memory.')

test_df = test_df.copy()
test_df['path'] = test_df['path'].astype(str)
test_df['label_id'] = test_df['label_id'].astype(int)

missing_files = test_df[~test_df['path'].map(lambda p: Path(p).exists())]
if len(missing_files):
    print('Warning: missing test images will be skipped:', len(missing_files))
    display(missing_files.head())
    test_df = test_df[test_df['path'].map(lambda p: Path(p).exists())].reset_index(drop=True)

split_counts = test_df['label'].value_counts().reindex(CLASS_NAMES, fill_value=0)
print('Test samples:', len(test_df))
display(split_counts.to_frame('count'))

fig, ax = plt.subplots(figsize=(9, 4))
split_counts.plot(kind='bar', ax=ax, color='#4C78A8')
ax.set_title('Test Set Class Distribution')
ax.set_xlabel('Emotion')
ax.set_ylabel('Images')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label


def make_dataset(dataframe, batch_size=BATCH_SIZE):
    paths = dataframe['path'].astype(str).values
    labels = dataframe['label_id'].astype(np.int32).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


test_ds = make_dataset(test_df)
y_true = test_df['label_id'].to_numpy()
print('Batches:', int(np.ceil(len(test_df) / BATCH_SIZE)))


In [ ]:
available_specs = {}
missing_models = []
for model_name, spec in MODEL_SPECS.items():
    if Path(spec['path']).exists():
        available_specs[model_name] = spec
    else:
        missing_models.append((model_name, spec['path']))

if missing_models:
    print('These models have no .keras file and will be skipped:')
    for model_name, path in missing_models:
        print('-', model_name, path)

if not available_specs:
    raise FileNotFoundError('No model files found in artifacts/models for the configured specs.')

print('Models to evaluate:', list(available_specs))


In [ ]:
results = []
predictions = {}
probabilities = {}
reports = {}
confusions = {}

for model_name, spec in available_specs.items():
    print()
    print(f'Loading {model_name}: {spec["path"]}')
    model = keras.models.load_model(spec['path'], compile=False)
    model.compile(loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    start = time.time()
    loss, eval_acc = model.evaluate(test_ds, verbose=0)
    eval_seconds = time.time() - start

    start = time.time()
    probs = model.predict(test_ds, verbose=0)
    predict_seconds = time.time() - start

    y_pred = np.argmax(probs, axis=1)
    acc = accuracy_score(y_true, y_pred)
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(len(CLASS_NAMES))),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))

    predictions[model_name] = y_pred
    probabilities[model_name] = probs
    reports[model_name] = report_dict
    confusions[model_name] = cm

    results.append({
        'model': model_name,
        'loss': float(loss),
        'keras_eval_accuracy': float(eval_acc),
        'accuracy': float(acc),
        'macro_precision': float(report_dict['macro avg']['precision']),
        'macro_recall': float(report_dict['macro avg']['recall']),
        'macro_f1': float(report_dict['macro avg']['f1-score']),
        'weighted_f1': float(report_dict['weighted avg']['f1-score']),
        'eval_seconds': float(eval_seconds),
        'predict_seconds': float(predict_seconds),
        'ms_per_image': float((predict_seconds / len(test_df)) * 1000),
        'parameters_million': float(model.count_params() / 1_000_000),
    })

results_df = pd.DataFrame(results).sort_values('accuracy', ascending=False).reset_index(drop=True)
display(results_df.style.format({
    'loss': '{:.4f}',
    'keras_eval_accuracy': '{:.4f}',
    'accuracy': '{:.4f}',
    'macro_precision': '{:.4f}',
    'macro_recall': '{:.4f}',
    'macro_f1': '{:.4f}',
    'weighted_f1': '{:.4f}',
    'eval_seconds': '{:.2f}',
    'predict_seconds': '{:.2f}',
    'ms_per_image': '{:.2f}',
    'parameters_million': '{:.2f}',
}))


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(19, 4))
metric_specs = [
    ('accuracy', 'Accuracy'),
    ('macro_f1', 'Macro F1'),
    ('weighted_f1', 'Weighted F1'),
    ('ms_per_image', 'Inference ms / image'),
]
for ax, (metric, title) in zip(axes, metric_specs):
    colors = [available_specs[m]['color'] for m in results_df['model']]
    ax.bar(results_df['model'], results_df[metric], color=colors)
    ax.set_title(title)
    ax.set_xlabel('Model')
    ax.tick_params(axis='x', rotation=20)
    if metric != 'ms_per_image':
        ax.set_ylim(0, 1)
    for idx, value in enumerate(results_df[metric]):
        label = f'{value:.3f}' if metric != 'ms_per_image' else f'{value:.1f}'
        ax.text(idx, value, label, ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
n_models = len(results_df)
fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))
if n_models == 1:
    axes = [axes]

for ax, model_name in zip(axes, results_df['model']):
    cm = confusions[model_name]
    cm_norm = cm.astype('float') / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    sns.heatmap(
        cm_norm,
        ax=ax,
        cmap='Blues',
        vmin=0,
        vmax=1,
        annot=True,
        fmt='.2f',
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        cbar=False,
    )
    ax.set_title(f'{model_name} Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
report_frames = []
for model_name in results_df['model']:
    frame = pd.DataFrame(reports[model_name]).T.loc[CLASS_NAMES, ['precision', 'recall', 'f1-score']]
    frame['model'] = model_name
    frame['class'] = frame.index
    report_frames.append(frame.reset_index(drop=True))

report_df = pd.concat(report_frames, ignore_index=True)

for metric in ['precision', 'recall', 'f1-score']:
    pivot = report_df.pivot(index='class', columns='model', values=metric).reindex(CLASS_NAMES)
    plt.figure(figsize=(max(7, 2.2 * len(results_df)), 5))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlGnBu', vmin=0, vmax=1)
    plt.title(f'Per-class {metric}')
    plt.xlabel('Model')
    plt.ylabel('Emotion')
    plt.tight_layout()
    plt.show()


In [ ]:
comparison_rows = []
for idx, row in test_df.reset_index(drop=True).iterrows():
    true_id = int(row['label_id'])
    item = {
        'path': row['path'],
        'true_label': CLASS_NAMES[true_id],
    }
    for model_name in results_df['model']:
        pred_id = int(predictions[model_name][idx])
        item[f'{model_name}_pred'] = CLASS_NAMES[pred_id]
        item[f'{model_name}_confidence'] = float(probabilities[model_name][idx, pred_id])
        item[f'{model_name}_correct'] = pred_id == true_id
    comparison_rows.append(item)

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.head(20))
